# Étape 1 : Extraction des Flux RSS

In [ ]:
# %pip install feedparser

In [24]:
import feedparser
url_alerte = "https://www.cert.ssi.gouv.fr/alerte/feed/"
rss_alerte_feed = feedparser.parse(url_alerte)
for entry in rss_alerte_feed.entries:
    print("Titre :", entry.title)
    print("Description:", entry.description)
    print("Lien :", entry.link)
    print("Date :", entry.published)

url_avis = "https://www.cert.ssi.gouv.fr/avis/feed/"
rss_avis_feed = feedparser.parse(url_avis)
for entry in rss_avis_feed.entries:
    print("Titre :", entry.title)
    print("Description:", entry.description)
    print("Lien :", entry.link)
    print("Date :", entry.published)


Titre : [MàJ] Vulnérabilité dans Zimbra Collaboration Suite (17 juillet 2023)
Description: \[Mise à jour du 26 juillet 2023\] Publication du correctif de sécurité par l'éditeur Le 26 juillet 2023, l'éditeur a publié un correctif de sécurité \[1\] pour cette vulnérabilité immatriculée CVE-2023-37580. Il est donc fortement recommandé de déployer le correctif P41 pour toutes les versions...
Lien : https://www.cert.ssi.gouv.fr/alerte/CERTFR-2023-ALE-007/
Date : Mon, 17 Jul 2023 00:00:00 +0000
Titre : [MàJ] Vulnérabilité dans Citrix NetScaler ADC et NetScaler Gateway (19 juillet 2023)
Description: Le 18 juillet 2023, Citrix a publié un avis de sécurité concernant plusieurs vulnérabilités. La plus critique, dont l'identifiant CVE est CVE-2023-3519, permet à un attaquant non authentifié d'exécuter du code arbitraire à distance. L'équipement est vulnérable s'il est configuré en tant que...
Lien : https://www.cert.ssi.gouv.fr/alerte/CERTFR-2023-ALE-008/
Date : Wed, 19 Jul 2023 00:00:00 +0000
Ti

# Étape 2 : Extraction des CVE

In [84]:
import requests
import re


cve_list_avis = []

# liste de dictionnaire pour le dataframe
list_cve = []
#url = "https://www.cert.ssi.gouv.fr/alerte/CERTFR-2024-ALE-001/json/"
for entry in rss_avis_feed.entries:
    link_array = entry.link.split("/")
    # example split :
    #['https:', '', 'www.cert.ssi.gouv.fr', 'avis', 'CERTFR-2026-AVI-0701', '']
    bulletin_name = link_array[4]
    url = "https://www.cert.ssi.gouv.fr/avis/" + bulletin_name +"/json/"
    #print(url)

    response = requests.get(url)
    data = response.json()
    for cve_item in data.get("cves", []):
            cve_id = cve_item.get("name")

            cve_data = {
                "ID ANSSI": bulletin_name,
                "Titre ANSSI": data.get("title"),
                "Type": "Avis",
                "Date": data.get("date"),
                "CVE": cve_id,
                "CVSS": None,
                "Base Severity": None,
                "CWE": None,
                "EPSS": None,
                "Lien": cve_item.get("url"),
                "Description": data.get("description"),
                "Editeur": None,
                "Produit": None,
                "Versions affectées": None
            }
            list_cve.append(cve_data)


    #Extraction des CVE reference dans la clé cves du dict data
    ref_cves=list(data["cves"])
    #attention il s’agit d’une liste des dictionnaires avec name et url comme clés
    # print( "CVE référencés ", ref_cves)
    # Extraction des CVE avec une regex
    cve_pattern = r"CVE-\d{4}-\d{4,7}"
    cve_list_avis.append(list(set(re.findall(cve_pattern, str(data)))))


cve_list_alert = []
for entry in rss_alerte_feed.entries:
    link_array = entry.link.split("/")
    # example split :
    #['https:', '', 'www.cert.ssi.gouv.fr', 'avis', 'CERTFR-2026-AVI-0701', '']
    bulletin_name = link_array[4]
    url = "https://www.cert.ssi.gouv.fr/alerte/" + bulletin_name +"/json/"
    # print(url)

    response = requests.get(url)
    data = response.json()
    for cve_item in data.get("cves", []):
            cve_id = cve_item.get("name")

            cve_data = {
                "ID ANSSI": data.get("reference"),
                "Titre ANSSI": data.get("title"),
                "Type": "Alerte",
                "Date": data.get("date"),
                "CVE": cve_id,
                "CVSS": None,
                "Base Severity": None,
                "CWE": None,
                "EPSS": None,
                "Lien": cve_item.get("url"),
                "Description": data.get("description"),
                "Editeur": None,        #
                "Produit": None,
                "Versions affectées": None
            }
            list_cve.append(cve_data)
    #Extraction des CVE reference dans la clé cves du dict data
    ref_cves=list(data["cves"])
    #attention il s’agit d’une liste des dictionnaires avec name et url comme clés
    # print( "CVE référencés ", ref_cves)
    # Extraction des CVE avec une regex
    cve_pattern = r"CVE-\d{4}-\d{4,7}"
    cve_list_alert.append(list(set(re.findall(cve_pattern, str(data)))))

print(list_cve)


[{'ID ANSSI': 'CERTFR-2026-AVI-0692', 'Titre ANSSI': 'Multiples vulnérabilités dans Google Chrome', 'Type': 'Avis', 'Date': None, 'CVE': 'CVE-2026-10927', 'CVSS': None, 'Base Severity': None, 'CWE': None, 'EPSS': None, 'Lien': 'https://www.cve.org/CVERecord?id=CVE-2026-10927', 'Description': None, 'Editeur': None, 'Produit': None, 'Versions affectées': None}, {'ID ANSSI': 'CERTFR-2026-AVI-0692', 'Titre ANSSI': 'Multiples vulnérabilités dans Google Chrome', 'Type': 'Avis', 'Date': None, 'CVE': 'CVE-2026-10991', 'CVSS': None, 'Base Severity': None, 'CWE': None, 'EPSS': None, 'Lien': 'https://www.cve.org/CVERecord?id=CVE-2026-10991', 'Description': None, 'Editeur': None, 'Produit': None, 'Versions affectées': None}, {'ID ANSSI': 'CERTFR-2026-AVI-0692', 'Titre ANSSI': 'Multiples vulnérabilités dans Google Chrome', 'Type': 'Avis', 'Date': None, 'CVE': 'CVE-2026-11239', 'CVSS': None, 'Base Severity': None, 'CWE': None, 'EPSS': None, 'Lien': 'https://www.cve.org/CVERecord?id=CVE-2026-11239', 

In [45]:
print("CVE trouvés alerte:", cve_list_alert)
print("CVE trouvés avis", cve_list_avis)

CVE trouvés alerte: [['CVE-2023-37580'], ['CVE-2023-3519'], ['CVE-2023-42119', 'CVE-2023-42114', 'CVE-2023-42118', 'CVE-2023-42116', 'CVE-2023-42115', 'CVE-2023-42117'], ['CVE-2023-20273', 'CVE-2023-20198'], ['CVE-2023-4966'], ['CVE-2023-50164'], ['CVE-2023-46805', 'CVE-2024-21888', 'CVE-2024-21887', 'CVE-2024-21893', 'CVE-2024-22024'], ['CVE-2024-0402', 'CVE-2023-7028'], [], ['CVE-2024-21762'], ['CVE-2024-21413'], ['CVE-2024-3400'], ['CVE-2024-20353', 'CVE-2024-20359'], ['CVE-2024-24919'], ['CVE-2024-6387'], ['CVE-2024-42008', 'CVE-2024-42010', 'CVE-2024-42009'], ['CVE-2024-40766'], ['CVE-2024-47176', 'CVE-2024-47177', 'CVE-2024-47175', 'CVE-2024-47076'], ['CVE-2024-9380', 'CVE-2024-8963', 'CVE-2024-9381', 'CVE-2024-9379', 'CVE-2024-8190'], ['CVE-2024-47575', 'CVE-2024-50566'], ['CVE-2024-9474', 'CVE-2024-0012'], ['CVE-2025-0283', 'CVE-2025-0282'], ['CVE-2024-55591'], ['CVE-2025-0282', 'CVE-2025-22457'], ['CVE-2023-27997', 'CVE-2022-42475', 'CVE-2024-21762'], ['CVE-2025-31324'], ['CVE

In [40]:
import itertools

cve_list = list(itertools.chain.from_iterable(cve_list_alert)) + list(itertools.chain.from_iterable(cve_list_avis))
print(cve_list)

['CVE-2023-37580', 'CVE-2023-3519', 'CVE-2023-42119', 'CVE-2023-42114', 'CVE-2023-42118', 'CVE-2023-42116', 'CVE-2023-42115', 'CVE-2023-42117', 'CVE-2023-20273', 'CVE-2023-20198', 'CVE-2023-4966', 'CVE-2023-50164', 'CVE-2023-46805', 'CVE-2024-21888', 'CVE-2024-21887', 'CVE-2024-21893', 'CVE-2024-22024', 'CVE-2024-0402', 'CVE-2023-7028', 'CVE-2024-21762', 'CVE-2024-21413', 'CVE-2024-3400', 'CVE-2024-20353', 'CVE-2024-20359', 'CVE-2024-24919', 'CVE-2024-6387', 'CVE-2024-42008', 'CVE-2024-42010', 'CVE-2024-42009', 'CVE-2024-40766', 'CVE-2024-47176', 'CVE-2024-47177', 'CVE-2024-47175', 'CVE-2024-47076', 'CVE-2024-9380', 'CVE-2024-8963', 'CVE-2024-9381', 'CVE-2024-9379', 'CVE-2024-8190', 'CVE-2024-47575', 'CVE-2024-50566', 'CVE-2024-9474', 'CVE-2024-0012', 'CVE-2025-0283', 'CVE-2025-0282', 'CVE-2024-55591', 'CVE-2025-0282', 'CVE-2025-22457', 'CVE-2023-27997', 'CVE-2022-42475', 'CVE-2024-21762', 'CVE-2025-31324', 'CVE-2025-32756', 'CVE-2025-4428', 'CVE-2025-4427', 'CVE-2025-49113', 'CVE-2025

In [41]:
# Find duplicates
print(len(cve_list))
print(set(cve_list))

2261
1731


In [43]:
cve_list = list(set(cve_list))
print(len(cve_list))

1731


In [78]:
cve_list.remove('CVE-2025-55182')
cve_list.remove('CVE-2026-9150')
cve_list.remove('CVE-2025-66478')

# Étape 3 : Enrichissement des CVE

In [ ]:
cve_info = {}
for cve_id in cve_list:
    cve_info[cve_id] = {}
    url = (f"https://cveawg.mitre.org/api/cve/{cve_id}")
    response = requests.get(url)
    data = response.json()
    cvss_score = "not found"

    # Extraire la description
    try:
        description = data["containers"]["cna"]["descriptions"][0]["value"]
    except:
        print(f'dropped cve {cve_id}')
        cve_list.remove(cve_id)
        continue
    # Extraire le score CVSS
    #ATTENTION tous les CVE ne contiennent pas nécessairement ce champ, gérez l’exception,
    #ou peut etre au lieu de cvssV3_0 c’est cvssV3_1 ou autre clé

    try:
        # print(data["containers"]["cna"]["metrics"][0])
        if list(data["containers"]["cna"]["metrics"][0].keys())[0] == 'format':
            key = list(data["containers"]["cna"]["metrics"][0].keys())[2]
        else:
            key = list(data["containers"]["cna"]["metrics"][0].keys())[0]

    except:
        pass
        # print("no metrics")

    try:
        cvss_score = data["containers"]["cna"]["metrics"][0][key]["baseScore"]
        # print("found")
    except Exception:
        pass
        # print("not found")

    cwe = "Non disponible"
    cwe_desc="Non disponible"
    problemtype = data["containers"]["cna"].get("problemTypes", {})
    if problemtype and "descriptions" in problemtype[0]:
        cwe = problemtype[0]["descriptions"][0].get("cweId", "Non disponible")
        cwe_desc=problemtype[0]["descriptions"][0].get("description", "Non disponible")

    # Extraire les produits affectés
    affected = data["containers"]["cna"]["affected"]
    products = []
    for product in affected:
        vendor = product["vendor"]
        product_name = product["product"]
        try:
            versions = [v["version"] for v in product["versions"] if v["status"] == "affected"]
            products.append([vendor, product_name, versions])
        except:
            print(f"version error {cve_id}")
        # print(f"Éditeur : {vendor}, Produit : {product_name}, Versions : {', '.join(versions)}")

    cve_info[cve_id]['cvss_score'] = cvss_score
    cve_info[cve_id]['cwe'] = cwe
    cve_info[cve_id]['edition'] = product
    # Afficher les résultats
    # print(f"CVE : {cve_id}")
    # print(f"Description : {description}")
    # print(f"Score CVSS : {cvss_score}")
    # print(f"Type CWE : {cwe}")
    # print(f"CWE Description : {cwe_desc}")
    # print("END CVE\n\n\n")

version error CVE-2026-9149
version error CVE-2026-9149
version error CVE-2026-9149
version error CVE-2026-9149
version error CVE-2026-9149
version error CVE-2026-9149
version error CVE-2026-9149
version error CVE-2026-9149
dropped cve CVE-2026-47895


In [77]:
url = (f"https://cveawg.mitre.org/api/cve/CVE-2025-66478")
response = requests.get(url)
data0 = response.json()
url = (f"https://cveawg.mitre.org/api/cve/CVE-2023-37580")
response = requests.get(url)
data1 = response.json()

print(data0)
print("\n#############################\n")
print(data1)

{'dataType': 'CVE_RECORD', 'dataVersion': '5.2', 'cveMetadata': {'cveId': 'CVE-2025-66478', 'assignerOrgId': 'a0819718-46f1-4df5-94e2-005712e83aaa', 'state': 'REJECTED', 'dateReserved': '2025-12-02T17:09:52.015Z', 'dateUpdated': '2025-12-03T18:04:08.459Z', 'dateRejected': '2025-12-03T18:04:08.459Z', 'assignerShortName': 'GitHub_M'}, 'containers': {'cna': {'providerMetadata': {'orgId': 'a0819718-46f1-4df5-94e2-005712e83aaa', 'shortName': 'GitHub_M', 'dateUpdated': '2025-12-03T18:04:08.459Z'}, 'rejectedReasons': [{'lang': 'en', 'value': 'This CVE is a duplicate of CVE-2025-55182.'}], 'replacedBy': ['CVE-2025-55182']}}}

#############################

{'dataType': 'CVE_RECORD', 'dataVersion': '5.2', 'cveMetadata': {'state': 'PUBLISHED', 'cveId': 'CVE-2023-37580', 'assignerOrgId': '8254265b-2729-46b6-b9e3-3dfca2d5bfca', 'assignerShortName': 'mitre', 'dateUpdated': '2025-10-30T18:26:19.400Z', 'dateReserved': '2023-07-07T00:00:00.000Z', 'datePublished': '2023-07-31T00:00:00.000Z'}, 'containe

In [83]:
for cve in cve_list:
    cve_id = cve
    url = f"https://api.first.org/data/v1/epss?cve={cve_id}"

    # Requête GET pour récupérer les données JSON
    response = requests.get(url)
    data = response.json()

    # Extraire le score EPSS
    epss_data = data.get("data", [])
    if epss_data:
        epss_score = epss_data[0]["epss"]
        print(f"CVE : {cve_id}")
        print(f"Score EPSS : {epss_score}")
    else:
        print(f"Aucun score EPSS trouvé pour {cve_id}")
epss_score

CVE : CVE-2026-31752
Score EPSS : 0.000150000
CVE : CVE-2026-10974
Score EPSS : 0.000870000
CVE : CVE-2026-26007
Score EPSS : 0.000090000
CVE : CVE-2026-45994
Score EPSS : 0.000320000
CVE : CVE-2026-11056
Score EPSS : 0.001060000


KeyboardInterrupt: 

# Consolidation des données

In [ ]:
import pandas as pd
df = pd.DataFrame.from_dict()